In [1]:
import numpy as np
from sklearn.metrics import accuracy_score, roc_curve, roc_auc_score
import uproot
import glob
import pandas as pd

In [31]:
classes = ['QCD', 'Hbb', 'Hcc', 'Hgg', 'H4q', 'Hqql', 'Tbqq', 'Tbl', 'Wqq', 'Zqq']
n_classes = len(classes)
label_list = [f'label_{cls}' for cls in classes]
score_list = [f'score_label_{cls}' for cls in classes]

full_path = "../../results/switchpart/switchhead_full_8h_4e_2a_8dim/pred_*.root"
#full_path = "../training/JetClass/Pythia/full/LinformerPairWise/20240919-184207_example_LinformerPairwise_ranger_lr0.001_batch512/predict_output/pred_*.root"

arrays = []
concat_arrays = {}
for file_name in glob.glob(full_path):
    with uproot.open(file_name) as f:
        print(file_name)
        arrays.append(f["Events"].arrays(label_list + score_list))
for key in label_list + score_list:     
    concat_arrays[key] = np.concatenate([arrays[i][key].to_numpy() for i in range(len(arrays))])

y_prob = np.stack([concat_arrays[key] for key in score_list], axis=1)
labels = np.stack([concat_arrays[key] for key in label_list], axis=1).astype(int)

../../results/switchpart/switchhead_full_8h_4e_2a_8dim/pred_TTBarLep.root
../../results/switchpart/switchhead_full_8h_4e_2a_8dim/pred_HToBB.root
../../results/switchpart/switchhead_full_8h_4e_2a_8dim/pred_ZJetsToNuNu.root
../../results/switchpart/switchhead_full_8h_4e_2a_8dim/pred_HToWW4Q.root
../../results/switchpart/switchhead_full_8h_4e_2a_8dim/pred_WToQQ.root
../../results/switchpart/switchhead_full_8h_4e_2a_8dim/pred_HToGG.root
../../results/switchpart/switchhead_full_8h_4e_2a_8dim/pred_TTBar.root
../../results/switchpart/switchhead_full_8h_4e_2a_8dim/pred_HToCC.root
../../results/switchpart/switchhead_full_8h_4e_2a_8dim/pred_ZToQQ.root
../../results/switchpart/switchhead_full_8h_4e_2a_8dim/pred_HToWW2Q1L.root


In [32]:
overall_roc_auc = roc_auc_score(labels, y_prob, average='macro', multi_class='ovo')
predicted_labels = np.argmax(y_prob, axis=1) 
true_labels = np.argmax(labels, axis=1)  

accuracy = accuracy_score(true_labels, predicted_labels)

print(f'Overall ROC AUC = {overall_roc_auc:.4f}, Accuracy = {accuracy:.4f}')


scores = y_prob / (y_prob[:, :1] + y_prob) # defaults to 0.5 for QCD (not used)

rejections = []

for i in range(1, n_classes):
    if i == 5:
        percent = 0.99
    elif i == 7:
        percent = 0.995
    else:
        percent = 0.5
    
    mask = (labels[:, 0] == 1) | (labels[:, i] == 1)
    filtered_labels = labels[mask]
    filtered_scores = scores[mask]
    
    binary_labels = (filtered_labels[:, i] == 1).astype(int)
    binary_scores = filtered_scores[:, i]
    
    fpr, tpr, thresholds = roc_curve(binary_labels, binary_scores)

    idx = np.abs(tpr - percent).argmin()
    
    if fpr[idx] != 0:
        rejection = 1 / fpr[idx]
    else:
        rejection = np.inf  
    
    rejections.append(rejection)
    
    print(f'Rejection at {percent*100}% for {label_list[i]}: {rejection}')

model_name = "Model Details" 
flops = "-" 
parameters = "-"

df = pd.DataFrame([[
    model_name,
    f"{accuracy:.4f}",
    f"{overall_roc_auc:.4f}",
    int(rejections[0]),
    int(rejections[1]),
    int(rejections[2]),
    int(rejections[3]),
    int(rejections[4]),
    int(rejections[5]),
    int(rejections[6]),
    int(rejections[7]),
    int(rejections[8]),
    parameters,
    flops
]])
df.to_clipboard(index=False, header=False)

Overall ROC AUC = 0.9877, Accuracy = 0.8603
Rejection at 50.0% for label_Hbb: 10928.96174863388
Rejection at 50.0% for label_Hcc: 4166.666666666667
Rejection at 50.0% for label_Hgg: 123.11480455524777
Rejection at 50.0% for label_H4q: 1888.5741265344664
Rejection at 99.0% for label_Hqql: 5633.802816901409
Rejection at 50.0% for label_Tbqq: 32258.06451612903
Rejection at 99.5% for label_Tbl: 16260.162601626014
Rejection at 50.0% for label_Wqq: 533.9028296849973
Rejection at 50.0% for label_Zqq: 400.4004004004004
